# Lecture 10 — Quantum Linear Systems (HHL) (UnitaryLab companion)

The **quantum linear system problem (QLSP)** asks for a state $|x\rangle \propto
A^{-1}|b\rangle$ given a Hermitian $A$ and a state-preparation for $b$. The
**Harrow–Hassidim–Lloyd (HHL)** algorithm solves it by: (1) quantum phase
estimation to write eigenvalues $\lambda_j$ of $A$ into an ancilla register, (2) a
controlled rotation implementing $\lambda_j \mapsto 1/\lambda_j$, and (3) inverse
phase estimation to uncompute. Post-selecting the rotation ancilla on $|1\rangle$
leaves $|x\rangle \propto \sum_j (\beta_j/\lambda_j)|u_j\rangle = A^{-1}|b\rangle$.

The cost scales with the condition number $\kappa = \lambda_{\max}/\lambda_{\min}$
and with the phase-estimation precision (the bit count $d$). `ua.HHLAlgorithm()`
returns both the quantum and exact solutions; we verify the quantum $|x\rangle$
against `numpy.linalg.solve` and compare HHL to the QSVT solver of Lecture 9.

Contents:
1. Setup
2. Introspect the HHL result dictionary
3. Verify $|x\rangle \propto A^{-1}b$ (fidelity)
4. Condition number and the effect of phase bits
5. HHL vs. QSVT linear solver
6. Your turn

In [1]:
import numpy as np
import unitarylab_algorithms as ua
from unitarylab.core import Circuit
import contextlib, io

# Suppress the algorithms' verbose logging so cell outputs stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield

np.set_printoptions(precision=4, suppress=True)
print("ready: unitarylab_algorithms loaded")

ready: unitarylab_algorithms loaded


## 2. Introspect the HHL result dictionary

`ua.HHLAlgorithm().run(A, b, d)` takes a Hermitian $A$, a right-hand side $b$, and
the number of phase-estimation bits $d$. We inspect the returned keys and the
solution vectors.

In [2]:
A = np.array([[2, -1], [-1, 3]], dtype=complex)    # symmetric positive definite
b = np.array([1, 1], dtype=complex)
d = 6                                              # phase-estimation bits

with quiet():
    res = ua.HHLAlgorithm().run(A, b, d, backend="numpy")
print("keys:", list(res.keys()))
print("Estimated solution (quantum):", np.round(res["Estimated solution (quantum)"], 5))
print("Exact solution (classical)  :", np.round(res["Exact solution (classical)"], 5))
print("L2 error                    :", format(float(res["L2 error"]), ".2e"))
print("Post-selection probability  :", round(res["Post-selection probability"], 5))

keys: ['status', 'circuit_path', 'plot', 'circuit', 'Estimated solution (quantum)', 'Exact solution (classical)', 'L2 error', 'Post-selection probability', 'Computation time (s)']
Estimated solution (quantum): [0.8019-0.j 0.6012+0.j]
Exact solution (classical)  : [0.8+0.j 0.6+0.j]
L2 error                    : 2.27e-03
Post-selection probability  : 0.32468


## 3. Verify $|x\rangle \propto A^{-1}b$ (fidelity)

HHL yields a normalized state proportional to $A^{-1}b$. We normalize both the
quantum solution and `numpy.linalg.solve(A, b)` and compute the fidelity
$|\langle x_q|x_c\rangle|^2$, which must be $\approx 1$.

In [3]:
normalize = lambda v: v / np.linalg.norm(v)
x_q = res["Estimated solution (quantum)"]
x_c = np.linalg.solve(A, b)
fidelity = abs(np.vdot(normalize(x_q), normalize(x_c))) ** 2
print("HHL   (normalized):", np.round(normalize(x_q), 5))
print("numpy (normalized):", np.round(normalize(x_c), 5))
print("fidelity           :", round(fidelity, 8))
# Phase estimation is finite-precision, so we allow a small fidelity defect.
assert fidelity > 1 - 1e-3
print("Verified: HHL |x> matches A^{-1}b direction (fidelity > 1 - 1e-3).")

HHL   (normalized): [0.8001-0.j 0.5998+0.j]
numpy (normalized): [0.8+0.j 0.6+0.j]
fidelity           : 0.99999996
Verified: HHL |x> matches A^{-1}b direction (fidelity > 1 - 1e-3).


## 4. Condition number and the effect of phase bits

HHL's accuracy is limited by how well $d$ phase bits resolve the eigenvalues of
$A$; the difficulty is set by the condition number $\kappa$. We report $\kappa$ and
show the fidelity improving as $d$ grows.

In [4]:
print("kappa(A) =", round(np.linalg.cond(A), 4), "\n")
print(" d    fidelity")
prev = 0.0
for d_try in [3, 4, 6, 8]:
    with quiet():
        r = ua.HHLAlgorithm().run(A, b, d_try, backend="numpy")
    f = abs(np.vdot(normalize(r["Estimated solution (quantum)"]), normalize(x_c))) ** 2
    print(f"{d_try:2d}    {f:.6f}")
# More phase bits should not hurt; the finest resolution is essentially exact.
with quiet():
    r_fine = ua.HHLAlgorithm().run(A, b, 8, backend="numpy")
f_fine = abs(np.vdot(normalize(r_fine["Estimated solution (quantum)"]), normalize(x_c))) ** 2
assert f_fine > 1 - 1e-3
print("\nVerified: with d=8 phase bits, fidelity > 1 - 1e-3.")

kappa(A) = 2.618 

 d    fidelity
 3    0.999996
 4    1.000000
 6    1.000000
 8    0.999999

Verified: with d=8 phase bits, fidelity > 1 - 1e-3.


## 5. HHL vs. QSVT linear solver

Lecture 9's QSVT solver and HHL attack the same QLSP by different routes
(polynomial $1/x$ via singular-value transform vs. eigenvalue inversion via phase
estimation). On the same system both should land on $A^{-1}b$.

In [5]:
with quiet():
    res_qsvt = ua.QSVTLinearSolverAlgorithm().run(A, b, 0.01, backend="numpy")
x_qsvt = res_qsvt["Solution vector"]
f_hhl = abs(np.vdot(normalize(x_q), normalize(x_c))) ** 2
f_qsvt = abs(np.vdot(normalize(x_qsvt), normalize(x_c))) ** 2
print("HHL  fidelity vs numpy:", round(f_hhl, 6))
print("QSVT fidelity vs numpy:", round(f_qsvt, 6))
assert f_hhl > 1 - 1e-3 and f_qsvt > 1 - 1e-4
print("Verified: HHL and QSVT both solve A x = b consistently with numpy.")

HHL  fidelity vs numpy: 1.0
QSVT fidelity vs numpy: 1.0
Verified: HHL and QSVT both solve A x = b consistently with numpy.


## 6. Your turn
- Build an SPD matrix with a large condition number (e.g. eigenvalues $1$ and
  $50$) and observe how many phase bits $d$ HHL needs to reach fidelity $> 0.99$.
- Change $b$ to align with a single eigenvector and confirm HHL returns that same
  eigenvector (scaled by $1/\lambda$).
- Compare the post-selection probability across systems — it is the QLSP analogue
  of the LCU/block-encoding success probability.

See Lecture 10 for the full HHL complexity ($\widetilde{O}(\kappa^2/\epsilon)$ in the
original form) and how QSVT-based solvers improve the $\kappa$ dependence.